# Keyword study 2: pilot

Run first (from the repo root):
```
python scripts/kw_generate.py --config configs/kw_week1.toml --model llama3.2-3b --datasets amazon --split test --limit-per-cell 4
python scripts/kw_score.py --config configs/kw_week1.toml --validate-sentiment
python scripts/kw_score.py --config configs/kw_week1.toml --model llama3.2-3b
```

In [ ]:
import json
import pandas as pd
from IPython.display import Image, display
from fairsum.kwsum.collections import read_collections
from fairsum.kwsum.config import COLLECTIONS_PATH, KEYWORDS_PATH, MANIFEST_PATH, POOLS_DIR, load_config
from fairsum.kwsum.store import read_jsonl

cfg = load_config("../configs/kw_week1.toml")
pd.set_option("display.max_colwidth", 200)

In [ ]:
gens = pd.DataFrame(read_jsonl(cfg.generations_path))
gens.groupby(['model', 'condition'])['status'].value_counts(normalize=True).unstack(fill_value=0).round(2)

## Throughput and length

In [ ]:
gens.groupby(['model', 'condition'])[['seconds_per_item', 'completion_tokens']].mean().round(2)

## One collection under all nine conditions

In [ ]:
cid = gens['collection_id'].iloc[0]
for _, g in gens[gens['collection_id'] == cid].iterrows():
    print(f"--- {g['model']} | {g['condition']} | {g['status']}")
    print(g['raw'][:900], '\n')

## Sentiment classifier check (gate: accuracy >= 0.90)

In [ ]:
path = cfg.results_dir / 'sentiment_validation.json'
print(path.read_text() if path.exists() else 'run kw_score.py --validate-sentiment')

## Pilot scores (small n: eyeball only, no inference)

In [ ]:
scores = pd.DataFrame(read_jsonl(cfg.scores_path))
cols = ['unsupported_rate', 'contradiction_rate', 'delta_p', 'majority_amplification', 'coverage_gap', 'count_abs_error', 'keyword_uptake', 'n_words']
scores[scores['status'].isin(['ok', 'truncated'])].groupby(['model', 'condition'])[cols].mean().round(3)

## Decision rules (spec §8)
- 7-8B below 0.25 gen/s: cut test N to 40 per cell before cutting a model.
- Compliance below 50% on a condition: keep and report it (a finding, as REFER's 8B failure was).
- Sentiment accuracy below 0.90: switch to `siebert/sentiment-roberta-large-english`.
- A gated model unavailable: substitute the same family's next open model and note it.